# בנייה עם דגמי משפחת Meta

## מבוא

שיעור זה יכלול:

- חקר שני הדגמים המרכזיים ממשפחת Meta - Llama 3.1 ו-Llama 3.2, בנוסף לחלופות הזמינות כיום המשמשות בדוגמאות הקוד (Llama 3.3 ו-Llama 4)
- הבנת תרחישי השימוש והמצבים המתאימים לכל דגם
- דוגמת קוד להדגמת התכונות הייחודיות של כל דגם


## משפחת הדגמים של Meta

בשיעור זה נחקור 2 דגמים ממשפחת Meta או "עדר Llama" - Llama 3.1 ו-Llama 3.2. דוגמאות הקוד ניתנות להפעלה משתמשות ב-Llama 3.3 ו-Llama 4, שהן חלופות זמינות כיום בקטלוג דגמי Microsoft Foundry.

דגמים אלו מגיעים בגרסאות שונות וזמינים ב-[Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **הערה:** GitHub Models ייפסק בסוף יולי 2026. הנה פרטים נוספים על השימוש ב-[Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) כדי ליצור אב טיפוס עם דגמי AI.

> **הערה:** הגרסאות הספציפיות של הדגמים שלשמו נכתב במקור שיעור זה (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) הוסרו מאז מקטלוג דגמי Microsoft Foundry. דוגמאות הקוד למטה משתמשות ב-`Llama-3.3-70B-Instruct` ו-`Llama-4-Scout-17B-16E-Instruct` כחלופות הזמינות כיום בקטלוג דגמי Microsoft Foundry, ולכן הן עדיין פועלות כיום.

גרסאות הדגם (כפי שכוסו במקור בשיעור זה):
- Llama 3.1 - 70B Instruct (נדחק)
- Llama 3.1 - 405B Instruct (נדחק)
- Llama 3.2 - 11B Vision Instruct (נדחק)
- Llama 3.2 - 90B Vision Instruct (נדחק)

חלופות זמינות כיום המשמשות בדוגמאות הקוד:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*הערה: גרסאות Llama 3 שדוברו למעלה נשמרות להקשר היסטורי ואינן זמינות עוד ב-Microsoft Foundry Models.*



## למה 3.1

בלמה 3.1 עם 405 מיליארד פרמטרים משתייכת לקטגוריית מודלי ה-LLM בקוד פתוח.

המודל הוא שדרוג לשחרור הקודם למה 3 ומציע:

- חלון הקשר גדול יותר - 128k טוקנים לעומת 8k טוקנים
- מקסימום טוקני פלט גדול יותר - 4096 לעומת 2048
- תמיכה רב-לשונית משופרת - בזכות עלייה בטוקנים לאימון

אלו מאפשרים ללמה 3.1 להתמודד עם מקרי שימוש מורכבים יותר בבניית יישומי GenAI כולל:
- קריאה פונקציונלית מקומית - היכולת לקרוא לכלים ופונקציות חיצוניים מחוץ לזרימת העבודה של ה-LLM
- ביצועי RAG טובים יותר - בזכות חלון הקשר גדול יותר
- יצירת נתונים סינתטיים - היכולת ליצור נתונים יעילים למשימות כגון כוונון מדויק



### קריאת פונקציות מקומיות  

Llama 3.1 הותאמה במיוחד כדי להיות יעילה יותר בקריאות לפונקציות או כלים. יש לה גם שני כלים מובנים שהמודל יכול לזהות שיש להשתמש בהם בהתבסס על ההנחיה מהמשתמש. הכלים האלה הם:  

- **Brave Search** - ניתן להשתמש בו לקבלת מידע עדכני כמו מזג האוויר על ידי ביצוע חיפוש באינטרנט  
- **Wolfram Alpha** - ניתן להשתמש בו לחישובים מתמטיים מורכבים יותר כך שאין צורך בכתיבת פונקציות משלך.  

אתה יכול גם ליצור כלים מותאמים אישית שה-LLM יכול לקרוא להם.  

בדוגמת הקוד למטה:  

- אנו מגדירים את הכלים הזמינים (brave_search, wolfram_alpha) בהנחיית המערכת.  
- שולחים הנחיה מהמשתמש ששואלת לגבי מזג האוויר בעיר מסוימת.  
- ה-LLM יגיב בקריאה לכלי Brave Search שתיראה כך `<|python_tag|>brave_search.call(query="Stockholm weather")`  

*הערה: בדוגמה זו מתבצעת רק קריאת הכלי, אם ברצונך לקבל את התוצאות, תצטרך ליצור חשבון חינמי בדף ה-API של Brave ולהגדיר את הפונקציה עצמה`  


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### למה 3.2 

למרות שזו LLM, מגבלה אחת של למה 3.1 היא רב-מודליות. כלומר, היכולת להשתמש בסוגים שונים של קלט כמו תמונות כהנחיות ולספק תגובות. יכולת זו היא אחת התכונות העיקריות של למה 3.2. תכונות אלו כוללות גם: 

- רב-מודליות - היכולת להעריך גם טקסט וגם תמונות כהנחיות 
- וריאציות בגודל קטן עד בינוני (11B ו-90B) - מספקות אפשרויות פריסה גמישות, 
- וריאציות של טקסט בלבד (1B ו-3B) - מאפשרות לפרוס את המודל על מכשירי קצה / מובייל ומספקות זמן תגובה נמוך 

התמיכה הרב-מודלית מהווה צעד משמעותי בעולם המודלים בקוד פתוח. דוגמת הקוד למטה מקבלת גם תמונה וגם הנחיית טקסט כדי לקבל ניתוח של התמונה. מכיוון ש-Llama 3.2 90B Vision Instruct הושבת, הדוגמה משתמשת ב-`Llama-4-Scout-17B-16E-Instruct`, אלטרנטיבה רב-מודלית זמינה באופן מקומי. 

### תמיכה רב-מודלית עם למה 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## הלמידה לא נעצרת כאן, המשך את המסע

לאחר שסיימת את השיעור הזה, בדוק את [אוסף הלמידה של AI גנרטיבי](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) שלנו כדי להמשיך לשפר את הידע שלך ב-AI גנרטיבי!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
